# Purpose of this notebook
This notebook demonstrates how to (a) score gain curves, (b) plot heatmaps of score, and (c) pick the most promising operation points for SNR improvement measurements.

# Imports
You don't need to change anything in this section. Make sure you have [xarray](https://docs.xarray.dev/en/stable/index.html) installed.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
                    format= '[%(asctime)s] %(levelname)s - [%(filename)s:%(lineno)d] %(message)s',
                    datefmt='%H:%M:%S')

import numpy as np
import xarray as xr
import os

import matplotlib.pyplot as plt

In [ ]:
# Make sure scoring.py is in the same directory as this notebook
from scoring import score_ai_twpa_c_gain_data, find_best_operation_point, plot_gain_at_operation_point, plot_operation_point_parameters
from electrical_length import set_units_on_plot_axis

# Load S21 data

## Load example data from AI
This subsection loads the example data provided by Arctic Instruments.
You can use this to see how the rest of the code is meant to be used,
and also to understand the format into which you should load your data.

We highly recommend running this notebook with this example data first, before attempting to load your own data.

In [ ]:
# Make sure example_data/beta/gain-data_0.nc exist in the same directory as this notebook.

# Here we load just one pump power, for the sake of keeping the example code below simple.
# There's data for a few other powers in separate files under example_data/beta

netcdf_engine = None # None means that the first engine available on this system will be used.
#netcdf_engine="netcdf4" # This would be a safer choice, because the example data was saved with xr.DataSet.to_netcdf(<filename>, engine="netcdf4"), but may not be installed on your system.

# This example data is for the ~10.7 GHz pump frequency operation point
s21_data = xr.open_dataset(os.path.join("example_data", "beta", "gain-data_0.nc"), engine=netcdf_engine)

# This example data is for the ~14 GHz pump frequency operation point
#s21_data = xr.open_dataset(os.path.join("example_data", "alpha", "gain-data_0.nc"), engine=netcdf_engine)

# This reference data is needed for normalization when calculating the gain. We recommend using the unpumped zero-flux S21.
# In this example, the reference data is in fact the same for the alpha and beta operation points.
reference_s21_data = xr.open_dataset(os.path.join("example_data", "beta", "reference-data_0.nc"), engine=netcdf_engine)

In [ ]:
# Inspect the coordinates and data variables of the loaded reference dataset used for normalizing the response.
reference_s21_data

In [ ]:
s21_data

## Load your own gain scan data
This subsection is a placeholder for loading your own data.

How exactly you convert your data to the xarray format specified above
depends on details of your data acquisition, but it is generally not too
difficult. Here we show an example of how to convert data from raw numpy ndarrays,
which are quite easy to export from almost any measurement automation/data storage framework.

To run the example code below, change the cell type from Raw to Code.

## Define names of the coordinates and data variables in the loaded datasets

In [ ]:
# These are the default names used in AI's datasets.
# If you prefer other names, change the string in quotes
# to match your names, which you can see under the Coordinates
# and Data variables sections of the datasets loaded above.

IFBL = "ifbl" # Name of flux bias current (or voltage) coordinate
FREQUENCY = "frequency" # Name of VNA frequency coordinate
PUMP_FREQ = "pump_freq" # Name of pump frequency coordinate
PUMP_POWER = "pump_power" # Name of pump power coordinate
PUMP_STATE = "pump_state" # Name of pump state coordinate (boolean indicating on/off status of pump)

MAGNITUDE_LINEAR = "magnitude" # Name of variable containing |S21| on linear scale
PHASE = "phase" # Name of variable containing arg(S21) in radians

## Convert flux bias voltage to flux bias current
If your flux bias is not yet expressed as a current, use Rb (flux line resistance), which you extracted from the electrical length plot, to convert your flux bias voltage to a flux bias current.

In [ ]:
#Rb = <resistance extracted from electrical length plot>
#s21_data[IFBL] = s21_data[<flux bias voltage data variable name>] / Rb

# Normalize the S21 data with the reference data

In [ ]:
assert bool(reference_s21_data[PUMP_STATE].max()) == False, "The pump should be turned off in the reference data used for normalizing gain data."

assert IFBL not in reference_s21_data.coords or len(np.unique(reference_s21_data[IFBL])) == 1, f"More than one flux bias in the reference dataset: {np.unique(reference_s21_data)}"

flux_bias_in_ref = float(np.abs(reference_s21_data[IFBL]).min()) if IFBL in reference_s21_data.coords else reference_s21_data[IFBL]
logging.info(f"Flux bias in reference dataset = {flux_bias_in_ref/1e-6:.1f} µA")
assert np.abs(flux_bias_in_ref) < 10e-6, "We recommend using a reference dataset where the flux bias current is zero."

In [ ]:
assert np.abs(s21_data[FREQUENCY] - reference_s21_data[FREQUENCY]).max() < 1e3, "The gain data and reference data must be measured on the same grid. This is especially useful for phase normalization before unwrapping phase."

def compute_gain(s21: xr.Dataset, ref_s21: xr.Dataset):
    s21 = s21[MAGNITUDE_LINEAR] * np.exp(1j*s21[PHASE])
    ref_s21 = ref_s21[MAGNITUDE_LINEAR] * np.exp(1j*ref_s21[PHASE])
    return s21/ref_s21

gain = compute_gain(s21_data, reference_s21_data)

In [ ]:
gain

# Score the gain curves

In [ ]:
# Feel free to adjust the scoring parameters as you see fit. What is important is ultimately application-dependent/subjective.
total_score = score_ai_twpa_c_gain_data(gain_data=gain,
                                        gain_min=12, # Gives points for signal frequencies where gain exceeds this threshold
                                        gain_median=15, # Gives points for median gain
                                        ripple_max=5, # Gives points for signal frequency ranges where pk-to-pk ripple is less than this threshold
                                        f_min=4e9, f_max=8e9,
                                        #gain_min_exp, gain_median_weight, ripple_weight control prioritization of different metrics. Feel free to adjust the defaults.
                                        freq_key=FREQUENCY)

In [ ]:
fig, ax = plt.subplots()
total_score.plot(x=IFBL, y=PUMP_FREQ, ax=ax)
set_units_on_plot_axis(ax.yaxis, 1e9, "GHz", decimals=1)
set_units_on_plot_axis(ax.xaxis, 1e-6, "µA")

# Find and plot the best gain curves in the data

## The top-scoring gain curve

In [ ]:
fig, ax = plt.subplots()
plot_gain_at_operation_point(gain_data=gain, op=find_best_operation_point(total_score), ax=ax)
set_units_on_plot_axis(ax.xaxis, 1e9, "GHz", decimals=1)

## Top 10 gain curves

### Find the best points and plot them over the score heatmap
If the heat map of score contains multiple local maxima, adjust exclusion_radius such that you get points from different local maxima. The larger the exclusion_radius determines the minimum distance between the points. Start from 20 and adjust up or down as needed.

As you see in the heat map, the grid could still be a little finer along the horizontal flux bias axis. Therefore, if this is a permanent installation, one final round of very fine tuning could be worthwhile, once the local maximum with the best SNR improvement has been identified.

In [ ]:
best_points = []
for i in range(10): # <-- Adjust here the number of gain curves you want to extract.
  best_points.append( find_best_operation_point(total_score,
                                           excluded_points=best_points,
                                           exclusion_radius=20) # <-- Tweak such that the red points in the plot generated by this cell span different local maxima of score, if there are multiple local maxima in the data.
                    )

fig, ax = plt.subplots()
plot_operation_point_parameters(scored_data=total_score, ops=best_points, ax=ax)
set_units_on_plot_axis(ax.yaxis, 1e9, "GHz", decimals=1)
set_units_on_plot_axis(ax.xaxis, 1e-6, "µA")

### Plot the gain curves corresponding to the best points identified above

In [ ]:
# The functional part of the code below is basically just the plot_gain_at_operation_point(...) call.
# The rest is figure formatting.

ncols = 2
fig, ax = plt.subplots(nrows=int(np.ceil(len(best_points)/ncols)), ncols=ncols)
fig.set_size_inches(12, 4*len(best_points)/ncols)
plt.subplots_adjust(hspace=0.4)

for i,op in enumerate(best_points):
    a = ax[i//ncols,i%ncols]
    plot_gain_at_operation_point(gain_data=gain, op=op, ax=a)
    a.text(7.7e9, 20, f"#{i}")
    set_units_on_plot_axis(a.xaxis, 1e9, "GHz", decimals=1)